# Qwen3.5-0.8B V4: Full attention FIRST → Progressive Delta SECOND

This notebook produces **two distinct models**: full-attention-only (6 replacements, retaining all 18 original DeltaNet layers) and an experimental progressively substituted Delta version (18 possible replacements, selected with rollback). Results and models are saved separately.

**Why:** V3 recorded ~59.4% holdout next-token teacher agreement despite faster prefill and exhibited repeating chat output. Replacing 18 DeltaNet layers before validating the six full attention layers obscured the source of the problem. V4 checks full-attention parity first, then ranks individual Delta layers on TRAIN data, adds them gradually, trains and rolls back groups that fail DEV fidelity.

**Splits:** official WikiText train/validation/test; holdout is sealed until final evaluation. FFNs/norms remain frozen. No CeNN. Text-only. use_cache=False; no streaming performance claims.

**Output:** full-only checkpoint, best accepted Delta checkpoint (or first rejected Delta diagnostic-only checkpoint), statistics ZIP and optional model ZIPs; separate Hugging Face gates for each model.


In [ ]:
#@title 1. Environment and experiment parameters
%pip -q install -U "git+https://github.com/huggingface/transformers.git@main" "accelerate>=1.0" "datasets>=3" "huggingface_hub>=0.34" "safetensors>=0.5"
%pip -q install --no-deps "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@a17f294d0144c2a09c41ace8b9fbea6108af4eaa"
from pathlib import Path
from datetime import datetime, timezone
import os, random, math, json, copy, gc, hashlib, zipfile, re, shutil, time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import HfApi, login, get_token
from IPython.display import display, FileLink
from tinycenn_lm import qwen35_window_anchor_v4 as wa

MODEL_ID="Qwen/Qwen3.5-0.8B"
SEED=20261018
MAX_TOKENS=192 #@param {type:"integer"}
DELTA_WINDOW=128 #@param {type:"integer"}
FULL_WINDOW=512 #@param {type:"integer"}
ANCHORS=16 #@param {type:"integer"}
CHUNK=64 #@param {type:"integer"}
TRAIN_EXAMPLES=320 #@param {type:"integer"}
DEV_EXAMPLES=64 #@param {type:"integer"}
HOLDOUT_EXAMPLES=96 #@param {type:"integer"}
RANKING_EXAMPLES=12 #@param {type:"integer"}
DELTA_STEPS=100 #@param {type:"integer"}
GRAD_ACCUM=2 #@param {type:"integer"}
LORA_RANK=12 #@param {type:"integer"}
LR_START=0.00035 #@param {type:"number"}
EVAL_EVERY=25
DEV_MAX_DROP=0.025
AUTO_DOWNLOAD_STATS=True #@param {type:"boolean"}
AUTO_DOWNLOAD_MODELS=False #@param {type:"boolean"}
UPLOAD_IF_QUALIFIED=True #@param {type:"boolean"}
HF_PRIVATE=True #@param {type:"boolean"}
FULL_HF_NAME="qwen35-08b-v4-full6-only" #@param {type:"string"}
DELTA_HF_NAME="qwen35-08b-v4-progressive-delta" #@param {type:"string"}
BASE=Path("/content/Qwen35_V4_FullFirst_ProgressiveDelta")
REPORTS=BASE/"statistics"
FULL_EXPORT=BASE/"full6_only_model"
DELTA_EXPORT=BASE/"progressive_delta_model"
DIAGNOSTIC_EXPORT=BASE/"rejected_delta_diagnostic_model"
for path in (BASE,REPORTS,FULL_EXPORT):path.mkdir(parents=True,exist_ok=True)
assert 8<=MAX_TOKENS<=512 and FULL_WINDOW>=MAX_TOKENS
assert 0<=ANCHORS<=DELTA_WINDOW and DELTA_WINDOW>=16
assert TRAIN_EXAMPLES>=RANKING_EXAMPLES>=4
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
assert torch.cuda.is_available(),"Enable GPU in Colab runtime settings"
torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda",torch.cuda.current_device())
ATTENTION_SETTINGS={"delta_window":DELTA_WINDOW,"full_window":FULL_WINDOW,
                    "anchors":ANCHORS,"chunk_size":CHUNK}
print("GPU",torch.cuda.get_device_name(),"Sequence: FULL FIRST → DELTA SECOND")


In [ ]:
#@title 2. Three official WikiText splits: TRAIN, DEV, sealed HOLDOUT
# Unlike V2, validation/test are not slices of the training split.
DATA_SOURCE="wikitext-2-raw-v1"
def clean_texts(split,needed):
    ds=load_dataset("Salesforce/wikitext","wikitext-2-raw-v1",split=split)
    unique=[];seen=set()
    for row in ds:
        value=" ".join(str(row["text"]).split())
        if 80<=len(value)<=850 and not value.startswith("=") and value not in seen:
            unique.append(value);seen.add(value)
    if len(unique)<needed:
        raise RuntimeError(f"Split {split} has only {len(unique)} eligible rows; need {needed}")
    rng=random.Random(SEED+{"train":11,"validation":23,"test":41}[split])
    rng.shuffle(unique)
    return unique[:needed]
try:
    train_text=clean_texts("train",TRAIN_EXAMPLES)
    dev_text=clean_texts("validation",DEV_EXAMPLES)
    holdout_text=clean_texts("test",HOLDOUT_EXAMPLES)
except Exception as exc:
    print("Official data unavailable; deterministic fallback for debugging ONLY:",repr(exc))
    DATA_SOURCE="synthetic_fallback_not_valid_for_publication"
    subjects=["A scientist","A library","An engineer","A hospital","A teacher",
              "A spacecraft","A mathematician","An orchestra","A historian","A botanist"]
    verbs=["described","studied","measured","compared","discussed",
           "reconsidered","observed","evaluated","investigated"]
    suffixes=["a new method using controlled experiments and careful analysis.",
              "the importance of verification and repeating the measurement.",
              "several observations that challenged the previous explanation.",
              "the distinction between efficiency, accuracy and usefulness.",
              "how practical design choices affected the final result."]
    needed=TRAIN_EXAMPLES+DEV_EXAMPLES+HOLDOUT_EXAMPLES
    unique=[(f"{subjects[i%len(subjects)]} "
             f"{verbs[(i//len(subjects))%len(verbs)]} "
             f"{suffixes[(i//(len(subjects)*len(verbs)))%len(suffixes)]} "
             f"Deterministic synthetic debug record {i}.")
            for i in range(needed)]
    print("WARNING: synthetic data is NOT a quality benchmark and HF upload will be blocked.")
    random.Random(SEED).shuffle(unique)
    train_text=unique[:TRAIN_EXAMPLES]
    dev_text=unique[TRAIN_EXAMPLES:TRAIN_EXAMPLES+DEV_EXAMPLES]
    holdout_text=unique[TRAIN_EXAMPLES+DEV_EXAMPLES:
                        TRAIN_EXAMPLES+DEV_EXAMPLES+HOLDOUT_EXAMPLES]
assert not set(train_text)&set(dev_text)
assert not set(train_text)&set(holdout_text)
assert not set(dev_text)&set(holdout_text)
print(DATA_SOURCE,"official split sizes",len(train_text),len(dev_text),len(holdout_text))



In [ ]:
#@title 3. Official Qwen3.5 text-only teacher; cache train and DEV logits once
teacher,tokenizer,source_root=wa.load_official_text(MODEL_ID,dtype=torch.float16)
teacher.to(device).eval().requires_grad_(False)
original_config=teacher.config.to_dict()

def make_record(text):
    ids=tokenizer(text,add_special_tokens=False)["input_ids"][:MAX_TOKENS+1]
    if len(ids)<8:raise ValueError("Text too short")
    return {"input_ids":torch.tensor([ids[:-1]],dtype=torch.long),
            "gold_next":int(ids[-1]),"text":text,"length":len(ids)-1}
train_rows=[make_record(s) for s in train_text]
dev_rows=[make_record(s) for s in dev_text]
# Holdout records are NOT yet materialized or evaluated.
@torch.inference_mode()
def attach_reference(records):
    for index,rec in enumerate(records):
        x=rec["input_ids"].to(device)
        logits=teacher(input_ids=x,use_cache=False,logits_to_keep=1).logits[:, -1]
        rec["teacher_logits"]=logits.float().cpu().contiguous()
        if index%25==0:print("Reference logits",index,"/",len(records))
    return records
train_rows=attach_reference(train_rows)
dev_rows=attach_reference(dev_rows)

teacher.cpu();gc.collect();torch.cuda.empty_cache()
print('Teacher logits cached for TRAIN/DEV only; HOLDOUT untouched.')
